In [ ]:
# NOTEBOOK NAME
# RadarNetCDFsaver.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

import zipfile as zp          # used for unzipping ppi files
from pathlib import Path      # used to play with pathnames to save 
from datetime import datetime # used to manipulate time :)

import wradlib as wr          # used for having fun with radar data

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

import h5py                   # used for reading .h5 files (Radar Level 1 data)
import h5netcdf               # used for converting .h5 files to NetCDF

# TAKEN FROM "Part4IntroductionToGridding"
import cartopy.crs as ccrs
import pyart
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker



In [ ]:
# SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks')
from leroi.leroi import *

In [ ]:
# FUNCTION
#pcolormesh but takes 1D X and Y coordinates for centres of the pixels

def pcolormeshC(x_centers, y_centers, z, ax=None,
                            shading='auto', **pcolor_kwargs):
    """
    Create a pcolormesh from a 2D array and 1D coordinate-center arrays.

    Parameters
    ----------
    x_centers : 1D array
        X coordinates of cell centers (length = number of columns in z)
    y_centers : 1D array
        Y coordinates of cell centers (length = number of rows in z)
    z : 2D array
        Data array with shape (len(y_centers), len(x_centers))
    ax : matplotlib.axes.Axes, optional
        Existing axis to draw on
    shading : str
        Passed to pcolormesh (default: 'auto')
    **pcolor_kwargs
        Extra kwargs passed to pcolormesh

    Returns
    -------
    pcm : QuadMesh
        The pcolormesh object
    """

    x_centers = np.asarray(x_centers)
    y_centers = np.asarray(y_centers)
    z = np.asarray(z)

    if z.shape != (len(y_centers), len(x_centers)):
        raise ValueError(
            f"z shape {z.shape} does not match "
            f"(len(y_centers), len(x_centers)) = "
            f"({len(y_centers)}, {len(x_centers)})"
        )

    # Convert centers -> edges
    def centers_to_edges(c):
        dc = np.diff(c)

        edges = np.empty(len(c) + 1)

        # Interior edges
        edges[1:-1] = c[:-1] + dc / 2

        # Extrapolate outer edges
        edges[0] = c[0] - dc[0] / 2
        edges[-1] = c[-1] + dc[-1] / 2

        return edges

    x_edges = centers_to_edges(x_centers)
    y_edges = centers_to_edges(y_centers)

    if ax is None:
        fig, ax = plt.subplots()

    pcm = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading=shading,
        **pcolor_kwargs
    )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    return pcm

In [ ]:
# THIS WHOLE BLOCK CREATES AND STORES NET CDF FILES FOR GRIDDED RADAR DATA

# the reference number for the radar location
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 25

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

RadarFileDate  = YYYY + MM + DD 

# THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH

# PATH NAMES
# ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/ppi/' + str(RadarYear) + '/'
ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_ppi.zip'

# place where the zipped file lives
ZippedPath = ZippedFolder + ZippedFile
# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZippedPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')


# loop over every 5 min period in the day
for houri in range(0,24):
    for mini in range(0,60,5):
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] # add a string of format hh:mm:ss for printing
        print('working on ' + RadarFileTimePrint)
        

        # READ IN THE DATA TO A PYART FILE
        
        # the path of where the UNZIPPED radar data now live
        RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi/'
        RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi.nc'
        
        # open the radar data 
        radar = pyart.io.read(RadarFolder + RadarFile, delay_field_loading = True)  
        # radar = pyart.io.read(RadarFolder + RadarFile, delay_field_loading=False)
        
        # # RETRIEVE SOME FIELDS IN THE RADAR DATA
        # preferred_fields = ['corrected_reflectivity', 'corrected_velocity', 'corrected_differential_reflectivity',\
        #                     'corrected_cross_correlation_ratio', 'corrected_differential_phase', 'corrected_specific_differential_phase']
        
        # field = next((name for name in preferred_fields if name in radar.fields), next(iter(radar.fields)))
        # fields = [field]
        # field

        # Get ALL available fields from the radar
        fields = list(radar.fields.keys())
        print(f"Gridding fields: {fields}")
        
        #DEFINING THE CARTESIAN GRID SHAPE
        grid_shape = (41, 301, 301)
        grid_limits = ((0.0, 20000.0), (-150000.0, 150000.0), (-150000.0, 150000.0))
        coords = [
            np.linspace(lower, upper, n)
            for (lower, upper), n in zip(grid_limits, grid_shape) ]
        
        # SETS THE WEIGHTING AND CREATES A DICTIONARY OF FIELDS OR SOMETHING
        # I DON'T REALLY KNOW
        # ??????
        grid_fields = leroi_interp(
            radar,
            coords,
            field_names=fields,
            weight_type="Barnes",
            Rc=None,
            k=100,
            verbose=True )
        
        grid = build_pyart_grid(radar, grid_fields, grid_shape, grid_limits)
        # grid
        
        xgrid = grid.to_xarray() # change this loaded grid into an xarray data frame so I know how to work with it

        # save the converted gridded data as a re-usable NetCDF file
        NetCDFsaveFolder = '/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
        NetCDFsaveFile   =  RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
        
        NetCDFsavePath = NetCDFsaveFolder + NetCDFsaveFile
        
        if not Path(NetCDFsaveFolder).exists():
            print('Creating Folder: ' + NetCDFsaveFolder)
            Path(NetCDFsaveFolder).mkdir(parents=True, exist_ok=True)
        
        xgrid.to_netcdf(NetCDFsavePath)